# DataPilot AI: SFT-разогрев Tiny Aya и оценка «до/после»

Запускайте ячейки **строго сверху вниз**. Каждая безопасна при повторном запуске: готовые результаты не пересчитываются.

**Перед стартом**
- **Kaggle:** Settings → Accelerator → **GPU T4 ×2** (не P100), Internet → **On**. Добавьте `datapilot-ai.zip` как Dataset (Add Input). Секрет `HF_TOKEN`: Add-ons → Secrets.
- **Colab:** Runtime → **T4 GPU**. Секрет `HF_TOKEN`: 🔑 Secrets, включить доступ ноутбуку. Архив проекта загрузится по кнопке в шаге 1.
- Примите условия модели `CohereLabs/tiny-aya-global` на Hugging Face тем же аккаунтом, которому принадлежит токен.

**Если сессия перезапустилась:** выполните шаг 1 снова, затем нужный шаг. Результаты в папке `runs/` сохраняются.

**Про данные.** Тестовые семейства (`paid-profit`, `revenue-by-country`, `payment-status-counts`) никогда не участвуют в обучении. В исходных train-данных нет запросов с двумя и более JOIN, а в тесте они есть, поэтому расширенный вариант добавляет новые обучающие семейства с другим смыслом. Они придуманы после того, как был виден пробел, это нужно указать в отчёте.

In [ ]:
# ШАГ 1. Подготовка. Безопасно запускать повторно (после перезапуска сессии тоже)
import os, sys, glob, shutil, zipfile, base64, zlib
_BUNDLE = {
    'dp_pipeline.py': (
        'eNqlPGtv21aW3/Urbm8HK7KRKNtJg4w8CtaN1Wl2XNtjOy0GtkHQEmWxpkiWpJy4qoA8ptvBJpjsLObTYgeLfXzbL66bNE6cBzC/QPoL80v2nH'
        'Pv5Uuy7HSMNpYu7+Pc837RnPPRv4/ejt6MTkbPRy9Hz8f32bIVW+uO68ds9Gx0Nn7KfmPt77t27ZbvWnt1Nno1Oh79NHo2fjx6ycZ/gC8/svED'
        'Nno3vg8bneAWNOX16JThf+9w+/HT8UP497EYfQvLTkc/qGkG57zk9AI/jFnLd127FTu+F6mhryLfU5/9ZDS01aeo248dN/nmtw7sOPnW3wtCv2'
        'VHybroKPkYO71kk37ous6eEdpf9+0oWf6NE3Qc1y51Qr/HAivuwhwmn63D11Lp87Xl5gprMH7L79qhvWLtRbXY8Y6q1pFV3Xf9Pcvlpa2Npdur'
        'ZnP1C5xYi3tBLQ4tx6va3iEvfd7c+HVzOXkSdeJqzw737TYvbTSXlpsb5p0NOiHwo3g/tKOv3Xqt1gYiBUgkM7Stth3W04G23fPl6D+6fstyu7'
        'Cw/vG1qwvpKl4qldp2B3CntXrtCmt17dZBYyvs2xUGUDVWfQ8+te625aev+44dNz613MjW6yUGP1xyzmug4RtglGO2Z0VdwQlA4Ifw6BXyA9Kc'
        '0a8T+PoMV8BXBgzxPXHcq/GjIpvAp5dZJqGvyF4w7Q0sejN+NDqD3ydis1e073PiIoQsAGSldDfW/cD2xC2jru266pb37FY/tvZcu8Fre45XQ/'
        'A5zInbfj9uZDe4vd6kcTsMs+ObW8trd7YqdObUH0Qk/M/8EPjWgE9O6HtGyw+ONF0gF/7XaT0cCVDvHcW2FYYWPKfRThsGA0OAZCAner58dLcL'
        '3xhepZ4A0Or2vQNYAYch+bUOXPna3C+v68kMp8M8kGuaWM8BvgcrDpIRBOdKQ8wrLiZWyC8GmVJA3g2d2NZoodG2W37b1ng/7lRvAGp5aAeu1b'
        'K5ri9m13TcPvChgDIw7lpOLL/E9j1EC845f6+ShI1YmFleG/YI7bgferiCfdBgcym0IHaRzTb6Hsp+Mwz9UOsU+Xj0Aj6SHiMOPBsdjx+MnxBn'
        'v0pYeJA9ZFhnA2Cw7fr8wtzucMczDGOAsG9X5z+em6vvDiWUYkUOvgpdUkpj4ASm40Wx5bqaFe5HkxKIBE8FEOYzOX8Rheqv/wd72aEH34+qPc'
        'uzQIv89YyhKMJlHoHWBuEZnYKUfce09eY6u379hi5E7wQlFiY9hXs/xLtWq8QSVaBTbPeqgdU6gO2iRMrgvkCaDh8gIVNhGrJqj2XgYtWv2QDv'
        'MhTLwlZFMnuieoQcKB1EOkZdl/5NKBy2iJpEYz5506qUsJ7txRzOx3MylC8czK4wft4l+c+A6gIey6IElRgDcgCzgX19MH5aFyjart9I2AdAzX'
        'KP4I+2HYNtNOGemmQCxzMPyDoLsUcTBcRwojjSeE08yfNesqDCkhm1u3544Hj7HK+T7mjDlcEotXwvRowiEB8CxtSPsv3zdcFBz0Y/Sr1/goY9'
        'M1NYGjg/MJUN00CJfiXvELVCJ0DChOVyuQTzWBXMcod9AM5Ar4fErh6yAMweu1lr24c1rw84XLj5D/OLLO7aHuxhBXF1H9b1AzBwNrDc1+zbb1'
        'kMsgIPl5uf3F5aNT/dWFvdaq4uNzzfc+BGoQVexqGdrE349YjWp8aW3WTJsaWOAwCGh07Lzs6AlWFcilzbDti10vrmb1caHBX/+q/XlzY3v1zb'
        'WG6k5plMsmm1e8ChdKtqlyVmmlXvsHQqTeKlqJ8cxqotcARoVbzUYjt8s7nSvLXF5hnc73MW7Juh78K0Lz9rbjQZfPasnt0oF7Ys73DOvmWwYY'
        'DyOY/Y2gFUTT8Ij7kFrshWk93ZbG4U4WObd9abG/RE3ZaVp14Xj32/y+A2YJdteR/4WrjP+96kLC+yvLS19MnSZjO9DFv7cnXycmWOnLjN+C+0'
        'XyBdWbWdWUJgK6hjdLr2W64VRVrZRta1w5Zt+CH4DVFZZ7c32eraFlu9s7LCdY46g4Oa2k2YuANuAoofa+/V5ubmzQg0T88yEHAaWTAj227j90'
        'XW9tk0cA7ZGriZGxtrG+bm1tp6Yx5xUu2wX3RwiWcj+07lyYzLeBFPylmTeChHNnruILSg+bSPdEZOs7x+CUV7Qg0LyRfKFhVCzsop/YrWP4ph'
        'oqYbUeA6set4oEB0UI+76gFh8zqfqYP56E+gr8C2ZwOcouI6Iwt5LPxMEcuAO1rJOLDFQOe0DnbkCiN9fRXVtfRjQlAyGl8XTIikQu/1BwlBir'
        'r07PHjRM+DGm91JQOYUXzk2ll92ersm6joAYcYg2gCczzZEzV2x9k3giMOflYPBmdPR224H1oUc9V64JOAuAYOLRfnVZgFi9Wx5Fua6LWgF5ts'
        'nx1WpONit+xFyDC3hAk/56mVISMZLeEhu22zhWEQfhEr4Z4RBVJ228Qgsc72fN+FOcg/Ox5P1iH4wIDTfPQ9v320Dc5kFMBmtglC2LNivtsY8P'
        'gosHmd48amv/cVsDavcAEsr1sUokrYNX8PjYLAoD5UrI68S0Cj10y3xkCAwJEj1gUM+7/ALi/BRXismO45OHIPkT2RZ87Ak/senNM/jh+iE0GD'
        '6Ov9BE4ruhbPR6/Hj8ffkXk+hjnfs/Hv0cdDV5d2gVXfiVkq0j9cWflcuCci3oKpGG79afRG8QIygiF9b42uV5G3vMLKKWmyRK2DfQyRcllUws'
        'wP5YjEaRaGHa9cYfPiSCSelTvSqpxHTbFVYwZ1pkZrDronbscA3wNi9/3ImLxEo5EDtk7wZ/FZoehAetGYGqFAAaJbicOHQC30wdG9xiueGzZe'
        'jiMlHJU8VAOO9hHmZMwY4SLDuuL3cDgVAPT56n8naBPCIk8EmiVETZQJxYtCbbSy2ir7wMqp1CJnvid7J0oWdEWM6iPI6lbwoVvdthOKMTUkA4'
        'ttrtwGTMnwXWDMNEMzMXerubllvs8CSiWZ6xtrX9yGKTSbo2EWemya/6zUDIZgh2Bmkekdr+OzXzHtKuB6QU9pCeY25JENzrQDprp2s3HVmF8Q'
        'v+ZqZRYc4Y5ANyP2ey5PbXIe9/8DSH1IkfIxWctT+Bc0kIqZT8cPiDRvadppQW1BXCPVSDbQ5VWblY3t2I7i3TLr9vf3QQQ7IOtmt783Ccg5pn'
        'HSuTgnND3Cg9AvwggCXMfYcsDdWeBZL2RKzJe6JAoV/w2iDLecUM51XpnpsxR8GhlmodJF9Z0myo4lrmL/wPZEeKdSSBCvaPyzT82ttd80VzEZ'
        'wnOgqSeCKmgK3lBoJ3YS533wwQcsnUca/w1Zm2coV3WhvdBdORk/gVtBvPqALnifwExWajDn+eiF8JfAlWFIfNrjbPxEBroPMBwkS/Vw/EQ/N4'
        '5cqKPx+mH8iBKDb4CPnhdjSKKL2e2AUsMMnAqCydEsMI7K0sJk+Ga2/bue61ttgdHwKJWLwgSNpBBQKp0o1GiAYUJd41wS6GjaMU8j6GDfa9kQ'
        '0TbpFwglsyIcm2ntc0q3w0d/EfzwTMgSGI53aPdfsQEBODTY6D9TP1TQqA7a9Rw6peQULi55oxinn4H78L1Qo0JHor8LNHiIlEDCLTJeAG1BJ1'
        '5hBNd9Ugan439GDkpor0T/VKZ03zLY8gGMI7xI2MK9EHpM5f6Ej47hWOSYR2Q1345eL7KrcK9HTCaVhYeO+mX0jm2A12kwbYAGSAMc64Zpog00'
        'zSGwmmANGM5JbuZsxGgebCHUTxIz0eq3LXPPagH924rhcnqGA0e0Hasa9Rx+geroYZLDBjfDCluojm/dWV5iXwjFXd+JPtJ22lf0HYN+CS0ykf'
        'sCPkPnsTfbc/z1+p1JoQaW+TdFcaAYCfUjkk60mjRAUquJek+dbUp3iFVvsiWQONeGKAEAgK94wNY1dm+hyB6FHy6LRhI8XHqra3n7NgvlCBIO'
        'h7euQTQBzIBR1XNSXO8QOvKblP6Br2Bf0Nyc4vc3xE7HQmliJHdGrEizZW5KGRykIiBfQwboGfuh3w+0eR1OzA4s6HqaTcYFN2HFPBjSOb0Yko'
        'DKrdCcyfkLFXZjyvxqFVDX6lYlK8GC+YUbsza5fslNrmc3mZ7f/jOJ6UuhK1TQ+4xQ/ZMw4SLri/IFHIAcwYg3B7jx9tzu0BCf5neHFaUoCBK2'
        'YMzPM6rI/IT0SLjulGR5fsG4nkiS8GAoYKMUZsbtCo5ELjkplA2pJAPGuuvLOA6tZj7BmZ9v+Ad8EmPBEY1IfFUUH+SFOqMcOkImR8+yGAMWm4'
        'IIZNqBIgRIRRk48gcSJcx+CuwgqhmFAIjY76Wu/CPTgHBX5/SyKgyIEqYR9uLQtrXkVsCe+54f2qaNlIyyTkjWgeofJtuc4/X0D9khJiQzCOOT'
        '6sn33CMT8G6FRxRoc+Mr3/Fgy2oVH1Xlo0Ew5JStwoz2LOHf5gHE+Ram07nX7wVH+CFqOQdOXHVBB3ryu3gQHFlwy7v00XFd8WnPiSPYAwtjEd'
        '+9pC9aKHcgQ+YcC2Bejfwt0n0Jx2MpGjl5/FBPPdVZKC2UOapVwa+AoKNhwhrwKYPXIbP2bS+O+x5ElPPGvHGVgftrTLi5eGyZ9sHspapXI08t'
        'gjMB8az8ZiA3Gk5kWofgxyJwmJTZKQsuRqcsa2gBUzvlRVZWROuUyT7STocOWiEDxNOLMLCLlAd1G5PlAahx9GQ+h8B4UVjVOHTVlM1Pt7YoDx'
        'MuqqHA7sSV9LK5QwUBd8p07k65Iq9imiqGMfEK36KRqaeP6abgdpltG/PvZOW1OV3nZYEySm5N0QnZSJLjQCmvH4RyOgCWM10A2kcglKFH8x7g'
        'I8yilg9dt8cwl2ADEtPhPoDjw15pUnqRiVoAWEcsQsxyC0qlL5Y2bi+tbm2CyA141Il5nYmKH36pAtQhphsooq8laeXq4bx4VCV9Sm6qy4eUlS'
        'AxQN/rGB0oVFPk/eUMJDE99mM8Z6IVQYUK6BpIlBxaoWN5MYXjkUwbyjHlBwkkapm8YoeDYY9qAzlxWLXaVgAMxIEtJYWozSGdIJsd1PPp2wg/'
        'XFemRBgRkD8bA6s8bFFDIzTmEAi724Hf6kYNsKzA3i0718wwJewvMoRwI4oBSGq9plq3kmwDgOdtpxVraQBRIQNrfnF78/YnK01zufnF7VvNzQ'
        'afA7jXf7e1tnHrM5NmLK2srN0yb62tftrg9j1SqCDlZmTvY50zqiMbSZ5GnSyRgNlFhY/UKEpaVJjAeQUISMLamEnrZDmhEiYrjt2WM3aTGcqI'
        '7ngfyp8iNyYUBZ+9fDgvKMmuSBY1WJYPy2j0xaEUtJZzDopaXB7qTJ3GU2BRXsipKPBjVl6yyRHQHKhuRUkiqu31HbdtAgMZQF2wf/04AId/gL'
        'sOMd8P82FYADegX0NeAJdzPZtUkN0guWYM4kTl02uSPKzGuPxoZqNQXTk+ej5DJ+1hozEZP4NiZo0Gz2c9Jy+bXJOQJi6ZXnq6QOPtSaLg+vR7'
        'yC+67d9zyamOLV5PSyDTJzo3RFpcZihEJuQdaj906bDmLs4f1qYAkuQu3sKmIoWiF9ic57fMlHIwBST3zCFAiFtyQ6qCIFKIDaYSFXfE+Jwy+x'
        'hkA1ivMPp5JGJlGa1WGEXX4Nkoh0e44cotR4Dk4Wg28/yABX7Ho1r5OYwp1AWS7P34sQhUgQTUUiNKYm8p6fUylz4CV24K72YdZaXGzvOSz+d4'
        'WmlKGgneV4x5LrsrHSBOvSS7X4Q6hWMXvsrJBrYNavwjI7I6dmx7EVwKTN8l5GH0Xzl8zxIN8gqfJMKRSYHUmcxF5qThVGRHTshrBmphu1GCiU'
        'nJQEX0L+R9noosPfqeKIy59GBad6WsqdjuvPzg1XpWBCb6SwBPfeAB9NHI/mK8o5pkpoWO+FxEmbhkMnS8VKgh6mUnoh73TLlVmAueFoVo83PV'
        'hblMnFFhhA34xijefKGfkyJPIjzJVrOuI4PmfOr/grCQVk+JCH9W9FMEhQkES9etA4JroiZSauOuE3dl164hfqFmjFjGb4kM2K2taZh0mdP1Yp'
        'AfYViASykk0CE4VxkH7MsxyWMPiSEqojxZYQiAPN/19zG7jk2iwjVFcA0YRQ/ybj5cmGgr3Z7kJPQ78USeHAbxc5sKZPCxa7kdTkM9616VJkAs'
        'TJEwtmryyqyAmuG6/aAPPnPPD4+qqAudbyxR4GN8zrjxsdgbr0e9rODR4k13Z28re17hzjP6XEnHXdqLTVx1bOekppEjdCtbAvUV6r3GM+cX5l'
        'R2rQ2T0J0HauBTA/8BVrii5mb6XrPPf5UsrGe1Lx5mQNjqwhQnIqnBsCqvRan4k+sWBu2vbXMcJ0xeQ3+8wBa7oPjBJqC2ESZBdhRjhClkRza2'
        'XqSwuVAfU50W2e6c5EEx7405f9TblMJ/JpWKzMK/JbeAMuw/gjbd8dBJxWuk0pIre+BPvufdgK/E0x3ejeOgXqulzeMDJNqwdjgv+kMwo6MIeD'
        'VvajPKc1olpIB+JCOFytrHemk2lsihyaWFFapeqIYIfIXhmTQ2pIJPKJf8evyU/KPX5EzJTDfL0zTJUcZ+oBQGsoRkzRzqiLPAJeg5nhUDBy6K'
        'EepbTrAyN70SVNgFg0zV65zi4qqCxj603D6eIQMy0mDSpifaBasEMosqYrmMhs1lNilAyBjQxDsAV5j2Gqb+FkKHgjFbgU7iZqq0ZyLI6aFwvv'
        'Ld4EkTElfPVA29MYs5k9m3lxsCzGlaT0yBgHqTMkXNZfOfNtdWG1xmasTjzVufNT9fMje3frfSzHd+qBlL67fN3zR/1+DTVXa28N9Ii/4zFGau'
        '+V72aiUVoEwv3HSnG5/0PdCCB5p+geFPQlzba3V7Vngg/N+WFdkRm0gziVAZnlMdm1H1PFfywfSwyjoMypjsa1PgrsY+aLAy9nCWZRBPnzPhJb'
        'Y6D1U1XjRyHrLyxtKXAtHltEb/cR7VE953Ls0mUyIOtYhnXImigNMTkDMHW4dNESQJvKssWP4R+l1K3rJddTicddGy/pl+QXN4QomJqPOMZSLA'
        'Ae6ZvkuA77kgcQysW0dasQkP9BJcI6KSY7DN4TNQFbPopTQa3YaopBe4EGW0+S5YUhoKXMvzcODyUL+jevN9qiTJjvZcUe451YYGuH05ObG8O8'
        'S44oUcl8eWk7cl0KyTce5pvqi3U4Bnkiulg65iHNPR7h0vSX6LuoQvm/+jhCC01UfsGhbXwNvS6GFd3O1DUVq8r6pjuWowWR1pW6h+c8Y0+o6v'
        'rr0QlJJRjqot1ChSeEltSFhLACf/WG6UC37U+03y/ZLSRZHde4CIsRleWeE3ufIw71FgqoKJeiu+UkVdBMeYahCFg+ejl2RpT1TOWFSRZxd8Ad'
        'LLGFo0bcUELr6GcGEa9+cnb2e2Dh0K0wTqEqEYEsDi/TyiSd+LCv2zOJR06LR9eLqtcdRtykJUxCqI/HFUxv3U/aPlsytTpiWKQ7YHbe9eLsFr'
        '/tzUbjHlU8hU5W3NZXMbk42E03lbtrpMyxnkc0LA7irnkLSr5NLLea+GWgSOhUzMbBNYKGSZkKCGFQRYHdYSfwujKZmiUOhN+wVwST3j50xJRh'
        'RJmPhSklRA0cIul3H/JCCiPAGq1QoVW09Po1wjvIgWM/QCUYUABsffYVKjmFQBmWtBlNPR/ELFRypk1QoqGqAGQ10M0zIx5k+MbCfReb8H1sP5'
        'xs7a1AmzllrXSxm3RdaxsM0m8x6xAYdZfTcmtxPzbCklEkuRIh2Wb/vb3AH7Z4Tk82i8yrFxdXtud1dxhS/2wL68RLlKXRBbLpXyyCT+QZq+hO'
        '9EMfbJ+I+8Ti/PoXHbLkd9qieVdwv2a1gbeMOig8lBhwu+VtVs9abC+DFYJ9kaS91idOhbkqNX0lKc0NvbMFXPQYCsZkdmD3Xk5cHImRJM57Zt'
        '9C73bJN8zfQEyzvSou0yJmhgdzDd5fzMMp0YkQykPKf/bDjybw+ncJRTF6JM217qBKDZfRFyj5/KiDNpnwEZAgv/evwADgn9PvCGwCi3XSuI7L'
        'bZA64snAIq09OHWYEitqmwQecCrjhECJHj4INwdjqgEvAJ8K3hxHYP9O5Qta5l1UE9YXokQoWaN5gmuuGYhrYHLbFOTRepIcK3BLHKQvykKm7S'
        'JqspU92BKcvOqfLlLLyq82bi5MlXV8j0wg0msu3n+NzZIBg14QAxgBFIVaW6x78Xb1qg/QiG+NrMRFFEUmjvyARMO+6R8EyzCqw0+6yMjUEqHE'
        'i60b6KctNhZmxwgEH64ZRMO5M9Rtj1+TrjsR2PXmNKPQE3SXB0/bsw5Lh91UWemJRGxtGqMNfpOXHjRuZt/z8Lg6tefzoW7Zx11YmZETn6ww+b'
        'v12ZfCcCE+jvqE1c2uSsS6zhzAqDaOchdZKp6pUaAv/gceYPBVDZ63vVd/xcbA82/TvyCR6KpjY9/bMA5zIS5nDzfQaXCORUaUD2bSdR2ugYWR'
        'PXUKVtMZsRQ+SjyZjLG6BZQVzWtuXym6BhpH4A9YK6k/bG+AYJV59d4ROTrzTYfDr0tYvmdDva5la436fmAjCBZL3hmegyn6ql6Q0EMZHePhGh'
        'mcg42CauTRsE7nZRcKaFc1gL7fDsnwZhA9CBGX29C6pPrCy3rH5ki1HUgwCUsgL30cJOBJ8Vams+L9QRYdwJdiKPzviEGG8jHA4GprtsABcY7n'
        'jwFNgbRFJDtKmufkKhSHL87f6fy/p2feH63G5WaFX5ZvL+50l+9oUuVbQboCeKuLkXgIuDITMc9Us4iX0rHY2kCeBtMtlqxX3LTaZm3jWkd95N'
        'UAfgI0WZUEx5ZDNjH/xo9A4o/AJbg3Gb+vsaIDGmf5CrJmWLtyrsQaUT9fwDDJzOK+RivzS+rN1Qe/SsA9uUo1oStCUdWFV5G+Tab5yAi1gs3z'
        '6YOr2TbJEU8cUJqCK0gSq6YbEJlb58CNa8xubnFq6p3xCb/MfoT3q+ciQnF51yjbQ31QjPVHeoLtj3R6oevhBOOtrRz8QbDuxTkLyio94PUH+Y'
        'sW92O5JKICGhMPVT9Tug4RBii4m/6fKvuZ6G6k2BnXeUkToVqhyQlo/KqCcYoRcdZeJP/Eh9CCp7xibPhKamDAy+IcRGf1El00y7f/L3hY7V33'
        'x5kdiRU6Z9iQ1+qaKf9TrIZ52lwBEsFTjATvRde7+3O+52MdrHV2ThExhXDZQ0Kb4ks6ZQj7lPmFOfUCn49k3meupdCXKKMMvyE2IJlY0/FOkj'
        'YAgm3segq1MkrE4Z1nPvJMv3dTAOGD+VTCgS6hgqVijDSx9FuiPZJSM9yR8wAo1WTWyjsJTT5qf5YtT2pUyr2QX5BwipSn9nc9DMlEKx4SRp+f'
        'kZyQEkeAtMs0wFa1mc4r+kyGXtIiNf8vdUhYhbSsnt+G7bDjXayGk3LthczCaUNlC/q2aj6ZAmFM/thaORHV8SVDT8oe1S0MDPKRLwyuXaVKUF'
        'oT91lRufiCTSeefvkkYrmX0nBnLhxazoAi6Zs9czYoos+RxXRAEmWHT84u99RZShyEA4hKbjERUaiXco6QH0nk2ivOHKuMx1xrAcFtVrtYzCM1'
        'p+Dcu/KRPphfefMkq3ft4O8vCItkrg00v/D5So9EE='
    ),
    'scripts/build_sft.py': (
        'eNqVV1tvFDcUft9fYfnJQ3YnIN62WhBVi0TVUlTy1M1q5Ox4l4G5YXshNGwFQQiqVFAeKqE+VJUqtepLtU3Zck34C56/0F/Sc2zP3pIiiJLM+P'
        'ic71x8Lh5K6eXzGy3zzEzMgTmo9syUmBfmOdGSJ3nLvDJT87K6aw7NPsHft9Udc1jtwvoV/L0G7k+45peStNBtkiZKR5pvpUKR1hkSC9WXyZZw'
        'pBNIEtuiP9IiUtdTXA6SPFFXwoZ5CireImD1vXle3aseVbvV3eoxsZqQCNpWTKr2PgINN8BaooXSBMyfAj/aZ/42kxoiJOYnoBwA0tS8AsouAY'
        'SJ+QdcRsZDUj2EJUgQ3LTiL2rtuNgH4ENixd8QEJuSTCjFh0JFg0I2G9UDNBLNsMGBxz5EcR8kn5lpkwAuAEzMX+DGIyTB6gVgA2Fa3QFXnSob'
        'c2REIyDCj6vvgO0+0g+cXdU98walbUj2CQNdU8DF4LhjA7b7aGS1h8bYbQglMoDIBCD2gpBS2kiyspCacDksuVSiXl9VRV6/S9EYyCIjJddX0m'
        'SLePIlWDbczoArzcskFHm/iIVUZAEFTzvyG5495sgNSRL2i3yQDGv2y0LrJB+qVa4k12IouU6KXIWlLHTRL9JaaDH+q4KKD4S+VXNunPv4808v'
        'r/LookhnBmP6bnElNpDYaDRiMSAuhaOREjGDTA3aDQI/SoicdEi3Z1egm+Q8EyTJIVwhZHLM05RJurnFzrbPf/XlF7c/+/LCxWBTrcFa9IssE7'
        'IvNsPgLOuea33NW99EvbWANgloaJJByoeqA0AXvDb8sfgd+wjT4qaQLJjtJYOZeucl4XnsSHmhkYz2zrFqD0JeliKPGXI6NCn0SDp2778ry4j3'
        'Mf5MCjVKtTdLi20NJuExh/EoKxVbPXHP36WyuKloL2gSkauRFBFX/STpnOep8orBhVTkDCEDcoacPnlybq7Xg49uG3Z6ZI3Qf+/8RhdN3qHoBW'
        '0T6iyGYFJI61Emcq2AvEN5riBsyGF+Wa0HYp5D/biGdtdMgAd0oMLx2MdBbPOsTAW7PoIGA6FoYgvqiyZxkfEhmRkDeZqVGpQtJuiqcNBcOhIK'
        'iQEqkAEEuzsQtdR6xJWCbspzjU5B0WjwCegLgXdGHBveca92YWuUpHHUhwRn+A+MwDyvMxq6cIcgHU5LDISEI7S9mbocL1OOGc+680AvtPgj0R'
        '6Pe0u+rZEFweVhcPSkHLlNIPq2tmwKrxbiO/AXJstRcCS20V0w0eWeP0h/wsrWdXOxtl100QiMwjwxa4m6juoccVGsD5v2VnNlXrnFSJcjTO6h'
        '0FxryeyR1Ixd51EvYCdO1JS5M705jIWvrdipmcF1hw/+upexE/lgu5ebgCV2W6d63Rq/FwRL/aNW4BMvg1nNfJ7ZOSPB43rmhOe8R5fsjm9rji'
        '3kcRzVHjPaaqGFmGz6Vik6OIKaMPcHHHqMXTGKnX191mFbN06FWCUpDd4J6914B64c5WpdDXTLXjzeCxS6h+THY14scojqFZGWHWrvHva+Y28D'
        'eLeYz3Hzs3lifjW/mx/Mn+YP89T86DoVXNGqBzDZmSrTRHesUQF15oANmMPeKvtAu5SPrJt4neVhx+rhy7xPNtBYCbbJpAWPFUuTXAS2IPANyw'
        'FhQ8saSsHjCDsmC0JrFPIAnO3s8BoqLZOS+bKdwfctXh/BHA3Y+11qEWiPdDqEWud8F8ojNBkEcVhYgdn4sLbYmC8WKCzfzwvL+oFe2GgriDLc'
        'oGDeH2v33D+rAUYghRubMwzGzQTmENzOqofVE3OwfMVdOFo6U+eCtNZxCK7mYLaij/B0yjBCs3giBXeA8D/9vzfLmtDVAeYM5HCYXYsTydxCdT'
        'bkyDZJbPrFNbsMjkjelAn0XRtASsOrBdT9wpSSxw4onOabPlDSXqHAIZ+GaZEP8ULfgR6yzdQoY3jwWXc2BXvuKDMr163HrqfO0VydSrhKsgE9'
        '+jkDc2GeUGPCbpwCisu1cUBu+9s83rndZ4Dnt9Bj2Mc7A162zUsCpfkaL+q2gF/6jySQfFvtopTzZ4wSiLdv697e4b+tN8n6Ojk9Jv7DCr8RYB'
        '+Lu9GALIwinAlRZFMsirC1RhF1Ke/6bOM/6cxcAg=='
    ),
    'scripts/sft.py': (
        'eNqNVltv2zYUftevINQXCZDV3BYUBjTASzIgm9cFTVpgKAqClmhbiyxqJJUmGwakyYo9tEC2p/0NL2iG3PMXpH+0c0j50q0eZsAwycPznQu/c4'
        '5d1+2KZx2y++Ue8faedX1S3VVjUp/Ux9W4uqiu65PqHtaX9Vn9lmwyzXbSTOg2qU/rX6vL6pbUv5gLt9W4PiEeHtZvQP1ddUW+2v32aav6ADBX'
        'eFadg8IFqS4JaOB+DLbu4fS6Gvvm+AGh6jNQua+unSxVmmrWy7girc9JwlUs0x63R3jCD3lcak7VDxlu+2meqmFIqj8A+QEcfIPI4Mt7Up0ThA'
        'Tw0+ov8OcOPEE3qj9NHOagPsMMdLaf0p3v/NB1XScdFUJqIpQjVMjzg1SKPFRcJ7zPykx77sbzzQ59sb27/UV3i25uvdje2Np1A+IuuT4hjwjk'
        'kzSBQuiQnoQfpDGnI1ZELiu1cDFqSN4FOgVyyOK9ydcN+nxrzj/Up8TEcWyAbkHD+5oNBpCCvTVyuOJj6JC5MSiOLQq+YFwmrL3sTGJgclAwqf'
        'hk/70SudOXYkQKpodZ2iONYAe2jpUk8NoQrZqINu2+UeN9PRF0hWQbIu+nAyvTMpuIgFdWEuByT7I059JxHEghGcHG89sOgU9BoqmPYUcOyhHP'
        '9Q7upOfbGyFLEsoakee2WugfZFsfFTxCtwPSPIzZea4sc/VY9XVLo9kQQ85cfxGaKHVR6v+FxxJWaC4XY41EwjN3pu9uiCGXvMt66rFO86MWO2'
        'KtQSZ6DBxagMELEQ/VxJ9+JpieAa4s0srkAo1l3lpb6C47bGU8H+jhRDnN540trT1ZpDqQLGmxOC5Hn1R98l8mleaF+qRaazkgQ55BlQCpb6Aw'
        'oU7fmbZyDAVyBXQ3i3Og+zHUxmVgOG9OL7Fs6mOyujCxCdoDs/FQQDGq6KVrUrW8jpXbm1ub5eqK+2ruISdiCw6wCohbhIa3aEUBW41Iitcoeo'
        'msC0EpUV4G3PdJX0iCK5LmRj9EGoeSs4Rqfqg9P1RFlmq8AmAk7ZvbodIyLTz/lQFvKhPwm5oMse4oNkwPDftk+nmEzeO+uoHvHaaKFHCz0Nh4'
        'YlhkXKcib9vWe2LyCR3pBDXq96hpJHNXjf3YlDSYn5a350wM2jqiSSoj8NkzIdozPyB5OaKmGqmld2TEdh1AQzikhhT2eLoNpuAFl7Rpohamx3'
        'Q8pCr9kUfAGWRjCs9MDSPLjKHH85B4wQpnmBlnMk/zAZVMc3stkwHJxGCAp1b7s4AodgCjRuOtwVHk5gI4ojhPorWVGRj6bEtpFoPdz3kXD3m8'
        'XwigPBiI9mTJAyI59kuqBSLnwM8ZJDYTCrNN0/3XCBr95FoOtxsC4ebnmUK/WF6PZhISRWSO4r3+v6Uz1huQpnKgw9PpS896vCej5XXMj2SUZc'
        'WQRasrzTaRooC3jpbCJShrMMI1BfdLmOAw8LKshVRmpkUxtU9N4bsbnee7nS7tftMUlbZDwrKrmRieSUKTUlwGJvQobmaLJUNTFlHzG8yHEM2t'
        'P7ITml/v4zPz1saQ908SN8mRKXaT6jczdR9s6cCwhyp6a/463dW/V3dt1/o50YXBB/VMac5GnFKTekpxDFLq2jloZ6LzN6P5s3g='
    ),
    'scripts/merge_adapter.py': (
        'eNqFU8GK2zAQvfsrhE4xJA699LDgQwj0lEAo27NQ4nGi1pZcSQ7NnrZbemkLpT9Q6B/sLht2aZrtL8h/1JGdeJNulwYClubNm5mneZRS98Nt3F'
        '11UX0hI/Vy0HOX7gb/v6sLt67Oibsm7grPt+7eXVef3RpvMPDeXZJOWjx73iVui9/DyauQYOKm+koQ6/M/IOgnhpBhORqNI0ppIPJCaUu4nhdc'
        'GwhSrXJScLvIxJTsghM8BnukVXq22MEgtS0Gv8cqgawJWc2lSZXOQZs9ZFBaVUNeKD3kpeHZaNytb0/VG5DiDHQQBAmkJOdCdsKTgOCvIHHbXT'
        'TQ8zIHaSf+pDthg4h4kjC+C3VorzflBmiXIBUvMxvToVqAhhGfmr4VctXjK96bZ2rKM/oUBU94YUEji10VEHsJHgj9qUN1KU3fpLbFhk+RqdIW'
        'pT3m0vC2FBqS+FSX0CRiisFpi6ge1jOY3Yi5lw1D/5Iw8oKzQgNqLiQkHZ8XeQmw4bpg/WRRmilu/XoksBQzYDkvYjorSnpcon3IR7w1okuM1U'
        '2J3dhhGOErz7FfmbBSYpXksOvI8CUc0rT5jSohMvIUGL6n4Jk441YoeSCK3e8GM6rUM2jWoa1OREqO2iF9Qh9yZkqmYh69NkrSMIJ3wljUlEBm'
        'gLQ61XWO9vDR7L7pvzvBwf83224htPB74L67q+qTW6Mp79y2+oY2RaN6e/5CK99gYIOWR4/eVx+rc3TsFq8QcYKLc0iKJsGhGZM8B8ZIHBPKmL'
        'cMY7TxTOOf4A+lJ6fO'
    ),
    'scripts/make_extra_cases.py': (
        'eNrdWd1v29YVf9dfccY9mEwoxnbXPgjQitR2Om9JHNgOhs0zBEaiUyYSyZCUW0MTkMRbk8LZvA4rNhTYhm0Pw14GRY0T1bGdh/4D5H+0c869Vy'
        'T1YcdJgAIzYJu899yPc37nm5qmJX9KTpJX+PsyOU4G6cPkIHmZPkmO073kAJK/JV8m/0r+nfwh+W/yn+QvyVeQvEh6ybOklz5KBqAj/QnTHyYn'
        'EIe26xkVwI16SR83OEwGJtAUkaX30730tzR5kD6EpA/bc2KFCTQNOPwgfZg+gfQB7nmAZwyIlm+1D/Pl94islzzFiw7Sz83S4vLa+vL1hXUTPl'
        '5duXkDPvoFroSFlZvX1y+t3bxmQfJPsSXu1mducOOD5Ah/v+XhftIDPbDdRjkI/S03NiF0th2v7ZRv7ZTrftuLwx0TAnun5XhxOYrtuB2J8chg'
        'Npg1lN0AWdtNf4+Xf5DuV0rpLs3yocjv6KGQPCPy5Buk+JZmj5DsQfIS7/s1svccp/do4nfM7UukOga+v8Qo3Sf50UmAEByk93HrF3hzu1H2ve'
        'YOkNRYkHQInXuMz3uQvEJKEush7WtpmlZyW4EfxmCHtwM7jBz1fifyvdJW6LeQ9fiTpnsL5MQNfC2JmYYd24Hb9GOr7ntb7m1FsubEsevdjkap'
        'Yt9vRopoEYdv2ZGzToOl0sLNtfWVa0ura1CFDe2y27A1E7Tlbduj/9fsHX5fs1vaZunG6srizYV1QfozZ+eWb4cNJvPbkSMePDf2Q3pcsG81HV'
        'zEKrG6vCRX/ZLmFpfo78cf0d8rq0hUKjWcLdiyW27TdSLdqJQAf7ZoyaZ4tOwgcLyGrmv1dhT7LScss/L4YcMJI9xoQ2dCJtaSf7ByKNNgqznE'
        '3xekGKziwPrQQ3R2WeOPgLYD0p5D0nEBHepox+t+CMnfGf4DiP3YblqamTtrbenq0sK6UH39ggGX1wQVXFlduQbifuDDT1eWr4O6ewR1WMFXy2'
        '1UfUuN1vACP//J0uoS+JZQ+OoM3WoGLl9fRGLPbjnVGbzQjGZkF/BD8MD1YIjkpmGMyUzY0xtJqyAPHhigygslRyJ0VdCpfw8yqluSKxRJfUwk'
        'dRaJ0r0JIkGv02jX43IbNTZiXTpLLMopov8DIQrUn/ukIkFXWfgzkggR9/Ni7AlPNEXpUHRfkQ9Cj57b5bn01ejC++xo8GCYtTIx88WniPny1a'
        'W1hSUdHbHuWvfathe78Y5hzrLkeWFO8jU3dlroH4T4h2Cg7H2SvWtJqgZoubOYWAoxgoDIA0EuB0/X5kBpczAGXUDQKV8zAbnYD8pDJZmm0l+z'
        '8AcUJp6h4Dt3uxMU+ST9DUVGdNQDIf6nAu/0Ma3rc/D45jz+Q1kBhgdiD01FypNww4MGwDbT50uIKMiBkcLzQxHJMPpQsAGUXtETTYRaOAUzb1'
        'gSwHfoffKwD6M9rTbF8bCyuri0SoPyvEXUPpMI4OryteV1kv4IxncJY33OhHkT3jPhR8YUnJWCCTOdhjIlNgrlnGUyxiN2OQ3hvHGz/OXkEb8h'
        '8DhBqRANFLRhAuTCMt8E8bxbmYi3MBtzxLDPtuqzDHUixERmBiMQi3MY4eBtEeaLyvxvCrYsDoHiIHmKiRiCV3CtdPSHUtZDApT0I06neyRcoj'
        'wU6SJlr49kYtpHmX/3Z8yiP+c8t1fwvuQNjjibQ/sWaeTTYUI7wKkTkS8bUltGlMQqSDTn3zMyutnpHl4I5mwXr/Tggk7g1ILQrTtsMLW6H8WG'
        '8vtiu3EVkUYvnXz1bChNeN+ED6YDyqE9ojxe+JExaInBQyll8nN5OMliHnDyfETJ2CThkrGTvb2WW+5EXSsz0WEow2P4mibvjma4Tx6E1EQ539'
        'ylJkJAwXHob3HLEaO84FoZGqdnPEV7RfvMQm6VT8nr0qiLRv5mshqMyTN3jG8jWEaMpSbzHY1ww5KBHuu2V3eaTQdXbMLF7xExupuZD84v2T6P'
        'VQHLxS//36eas9PokgFhrMQi8FG+iE5ORszw/1cLssTKF6ucRs2O4cdVmEH5nEdBGkJB5mfnPyjPzZZn50g31Nvc+9oksx+WZBFWoKhBZVm2n1'
        'lpCIdLwfFZ+mWG8SuEvMc4PqfSXepUDgrSqHdapqkLY3ycnAUHWRb8mrnUOFzBEC4pp7cv7FQVI0ohrJ3HJP5HGf84S8qqNq5JyJwxz9mljlBO'
        'nOke9Y3Qol/x9GNOo7PURNQ75ytwFDDDxg5FUywj2VH0RYOKniYbolw1EVDViMrqQYFoHqAJkOaQqnKmen4bfIcVUGaRQy7OWRQpRRARmPeYoA'
        'oISmZ/rBD3GT9pd0VAKQNmdTlAt9rL+l4FRchhKyud6VY3hCqT6uSK5Xy5qzKuNy4o+bRygHLj08fk9tfRkDOaf06NP0KbC8loPgMcgOivcOOR'
        'zSW388GUtIVv+xYBSwWqvCcU8j4jM6RoQsxmsURNFOu/2lhMeZ1aYBhDhAY33Sg+M2+U1bWIdae1rTg+CHm+aZA3C4H1rfpWmbs/NRJPdv6hE7'
        'dDD7Zkz7Rlu57ql9oBVIcNZetyeLtNMe0GvYW6IUksu4EcyDldK5e3KcLHO4FTpRazCbir3W7G/KZr1EW+5NT9FvKAVcX2nEUd6qZmTN3Pb8dB'
        'Oz7XnmXnszi0y/w9YmT/8HZETAUWM0UHRZIV0dWuFhvauuqB63KD7TlqItOeVtO3G5HedD1MxEi+9EQipj0tZIy6+LUYr6IbVhQ03ZgIcCNwt5'
        'gWfXjoBrohOtKfOM0GbV3P2o14FpLWNzRerW3CD6qgMVNatqa2Zbfo2h2k44b3DhIOtyCKbkZ7TxLeaztR7PreKaTRvaYkDp0tVFNM5mls8orQ'
        '/xRdiNMK4h3usZswW1JqJy5lwrYdujalRLhytDPP0ESoVrEkB8+P1RGCQ9trjMxN5nh7rmuipY19HTqBjiDuAmYiz6ldoz5R8dcrrehhTNCVkE'
        'xAvg3a2/FQK0M7dnTFjAlzORZybKjFBUbuMRck2fwovmNKzPlNH10IfY/j72Tc2EFV374U415a4ZTQiVD/UdastZbzmVNvxwyQzpcd+/khcP7V'
        'o8SYWwQHsoVQ/DRFn+j68jvUMag2JgUq+ij2WIRtdnDpFxSD0t3CtYQGXKyC3nQ8XVwSFQi1Q9s0oFqFWXRyUByn4Y1OvQKz3QxFRVL3m+2Wh1'
        'SbRpF/XKp8fUdDV1dBzBXA5c7dyux8o0uVhjCdirIbHJEqUxkqZmYNFcgwH1H7CsHWlU5EQEwsdsIxYwr5/ni/LnPMgsA3A7fMZ6fpPob3Lwhl'
        'zoJIQ0X5gq972tBZWcL9WZ+GGEKFN9E0646PXpqdUKPdCiI9ROPzonaIDi2qu271it2M0CtdBO1Xnla8lfRjtAO5jY1wzMCtNsoVHbzIHHVNhC'
        'PNKO4jPBDWoUiCoTT7RMqCLmcflivQGcqA83x+zTwADo5/ysV9fk3l3C6bKG57iTpnXLnLD7H8pZSG0ieiRKAL4Fmsgl1cnTcY8gcwf5E5QZqo'
        '3dLvUPoxzzzdIZ5YIAYtRFKVkkjaGfU+w9xvzBQkNjOCu0H9rlIJXXetRqG5ViM90Go1iq21mib8hQi0pf8B/L1epg=='
    ),
    'data/ecommerce-v1-extra-train.jsonl': (
        'eNrtnFFv2lYUx9/3Ka7ykqQCFGwnD5NYlSa0y9okE6Sa1hfkgjtZItjFphKqJiXptlZKtWiTpkmTpmnfIM1Ky9KEPuwL2N9o59x7DTbGGIIBp/'
        'EDAdv3Xl/+v4Pvuefek+cLamXhc7JQbhimtq/U07qsVtJavaLUjfRKdiFFFgy9qppYxqzLag3PPJH31WozqBqWeNpQDFPValjG+ts6tzrWB/s1'
        'vhPrvXUKH07hvWOdEfsQztmH9pF1ah/ZL/DQuiDYHLFfECj4wWpbLesSC5B1tSLfJtZf2JzVIqZmytUM3q+uPFHqSq2slIynVbxpMf8gv7FHNn'
        'Yf7uwt3Vom60VWmtwt7G4T1lGika92t3aI8yUMUia7cJhRKzkt45wtQU+++TJfyBMtY5iy2TByi9i9RbK+swmFa/K+klvEni0ufP/Z86F6CjHT'
        'c+uZXIunntizcD3FmOm5LTdjap/Ys3A9pZjpWZT34ykndMyjptaomfXmGM9NT4XxFPRoRE+0rfdUUvsACl3ax+T+o/nJVs7wL5dbvP9omEjCnE'
        'XazMdCpM38MJHEOYt0704sRLp3Z5hI0pxFuluIhUh3C26R9LpWaZTNdKOmmgZ7Xoc/l/yVwsRqWW9BkEurbf9EmEDwYAdhyH2l+ViT6xVifYTj'
        'DhQDsbDGmVvhU/vHIUMCyPobHIL+7lbeEWioZR+loC14P8C7k5VMDwHtfQiC9Qf54kZ+qfhwe0nNPG3INVM1m8upFUqFNuCiUlJNZd8gKkPTBQ'
        'VcNOSiZnipCivAVTSIjkV0VoSfHD6+6Hx8ccQLAyrMEOi21jCUhOYVaFLlwlCKM0UJbWj1BOaVYFLtwnBKM8S5IT+uJr/Mq8CkyrlRmpqe7g69'
        'o/vzg6r5cP5BaYArQUDeDskOcC069g/wOrAu4DSl8YYZgP0Ka50Bo5b1zziTKscxsU8IfmNwXrisiBFu1CbUizmjncAG4BNtzj6Agzb8fWH9C0'
        'YB7YOg3unZUPJsppRy+zqcZ4RTsnuF3Ydfkzvf0hopdkuyW9jMF/Akv8cmGGAKC5AHW9tbeyQbiluYBm4hwT0X3EIobnEauMUE91xwi6G4pWng'
        'lhLcc8Et9eF2XALuVo02cnsrBaEGD4qP2y7ni4Luc7yCMLv9NwqBX7ygR0AfLsDrBE94TGIAd+Z6XQW723McCp35SKk+zy3cbQvzyrqc8VJK7+'
        'PM2qaY9eAxux+zED1mIcE8c8xCCGYxesxignnmmMUQzFL0mKUE88wxu8dmektk9kQ1RxiV3cUD0FI9GMa29cY+RnqeiAfJ3uZSdy+D0C9BW3B3'
        'UFssd05YjAXOvrRaHESH/Pc7WQKuwMa5Bw+JoBuH/twlOmaEWsIbXq8Dpy+AH64MvF7mxtJnIxl3oKV3CXszPNTCpBg91uKgv7WEbEp6XS0r9J'
        'dSKmuGuewEYFizfqvgHhqPtuSygSSFmZAUEpIRkRQCSYozISkmJCMiKQaSlGZCUkpIRkQyeJxcnQnJ1YRkRCRXA0muzYTkWkIyIpJrPpJ0p4aR'
        'ftxMs7DVyF5sf0UfXfzG51xqjMu5ieI84RCmAhfWBW6PG6QwTnBwljFyGDHTm5l019PgPrSfKdo8zD5OcNqExuJEC129GgoFV+q6AUJoum8uck'
        'vN9PgM3wLjnabAtKS3/pejdwmJI9IyvQAiHI0AVbiOUJVaRa199+lyZd9vUrTiNURbloFEtap8uj/a7jecFK90TR/HKfdK0Qc6Al/axxi4gqEY'
        'R2/6fgKVibAirKWzKzD04Kh5Dvf5mQ/vLbpN5NLqfOIPd9y8obGSSqUkm+SLHFnsyTKpDa3eHBvKriY2NMCGsquj2lA3F8JQTBOeX2ldbu4rNX'
        'OsvKn+umF7wJhjjlHjt/YvPcofAfopRfgOXjxu7CaCNjWdXCqn50QP2LWl9+3aGnV9WO9i4yKNmHHlxyFcAxxRpWLNC0dgwpYfh3gNcESVyTUv'
        'HIH5Xn4c0jXAEVEi2Lxo9KWLOVt/WeaGqhhjZGZ06/jE/5XHd+iiZy8/hW79RUfEOgOX4cijrH0Mw/k5+CIf6eVXdD9Tb63RlcMx3mZiBxRPTU'
        'nh8mrLes/8nDO8AbOMwc4ErzUU8OZWcW9rh25N4sUZYTewAYhd5HJ0F9FofkTUWR09n6Lb/eHmIcTSPHhGSGIbkeWIjG0YYkwNg+eXJKYRYcbJ'
        '2MYhxdI4eLZKYhqR5a+MZhgsdEGvj+FvuGoNMA6A1PP4qIkcUJ7c0/MCxp1T1IBa9ks42YLPhxiT8PsbLuB8g3O4s9fl15N68Ebl8TY+Of5deF'
        '6mR18hlvrSQSa24gamSXqUFWOqLH1Gx1jbwKxFj7pSLNWlj7nYautLIqRtpXXQh6kU/rTtq+FT9M/+KK53o2NgJJeNhp49AO5F+DZh/+AAA9Vv'
        '6XDrarcVEAamfY0gDOyEf91zdUYhZHEe47VZV5DWOe3NFSl5grU+JMJ0kQg3D4kwKRJxukjEm4dEnBSJNF0k0s1DIo2JpBt8ZUSqqmGOs77Uqx'
        'W6RsmTCdmS2LB/gsPWjpjIky4EpjwLcRP9HxzXslD4mp1bTWHOauLSSUzVpKs646kpzllNXPmIqZp0UWY8NaU5q1mU92MqJq6p+LT8H0dHDcg='
    ),
}

IN_KAGGLE = os.path.exists('/kaggle')
WORK = '/kaggle/working' if IN_KAGGLE else '/content'

def find_project():
    for root in ['/kaggle/input', '/content', WORK]:
        for p in glob.glob(f'{root}/**/datapilot/api.py', recursive=True):
            return os.path.dirname(os.path.dirname(p))

src = find_project()
if src is None:                                   # Colab: загрузить архив вручную
    from google.colab import files
    name = list(files.upload())[0]
    zipfile.ZipFile(name).extractall('/content')
    src = find_project()
assert src, 'Проект не найден: загрузите datapilot-ai.zip (Kaggle: как Dataset через Add Input; Colab: кнопкой выше)'

PROJ = f'{WORK}/datapilot-ai'
if os.path.abspath(src) != PROJ:                  # чистая копия проекта, папка runs/ (результаты) сохраняется
    keep = None
    if os.path.exists(f'{PROJ}/runs'):
        keep = '/tmp/_runs_keep'; shutil.rmtree(keep, ignore_errors=True); shutil.move(f'{PROJ}/runs', keep)
    shutil.rmtree(PROJ, ignore_errors=True); shutil.copytree(src, PROJ)
    if keep:
        shutil.rmtree(f'{PROJ}/runs', ignore_errors=True); shutil.move(keep, f'{PROJ}/runs')

for rel, data in _BUNDLE.items():                 # встроенные скрипты, данные и модуль конвейера
    dest = '/tmp/dp/' + rel if rel == 'dp_pipeline.py' else f'{PROJ}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    open(dest, 'wb').write(zlib.decompress(base64.b64decode(data)))
sys.path.insert(0, '/tmp/dp')

HF_TOKEN = ''
try:
    from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
os.environ['HF_TOKEN'] = HF_TOKEN or ''

import importlib, dp_pipeline as dp
importlib.reload(dp)
dp.bootstrap(PROJ)
print('\nШаг 1 завершён. Проект:', PROJ)

In [ ]:
# ШАГ 2. Обучение SFT двух вариантов (~15 и ~25 минут, плюс установка окружения при первом запуске). Нужен GPU.
#   sft       : только исходные train-задачи v1
#   sft-extra : v1 train + 51 доп. обучающая задача (соединения 2-3 таблиц, GROUP BY; тестовые семейства не затронуты)
# Готовые адаптеры не переобучаются. Только расширенный вариант: VARIANTS = ('sft-extra',)
VARIANTS = ('sft', 'sft-extra')
dp.train_pipeline(PROJ, variants=VARIANTS, epochs=2)

In [ ]:
# ШАГ 3. Оценка на отложенном test-наборе через vLLM (при первом запуске vLLM ставится 10-20 минут).
# Baseline и уже оценённые варианты повторно не считаются.
dp.eval_pipeline(PROJ, WORK, variants=VARIANTS)

In [ ]:
# ШАГ 4. Сводка, разбор провалов и архив результатов
dp.compare(PROJ)
print('\n=== провалы расширенного варианта: что писала модель ===')
dp.show_failures(PROJ, 'sft-extra', limit=8)
archive = dp.package_results(PROJ, WORK)
if not IN_KAGGLE:
    from google.colab import files
    files.download(archive)                       # Kaggle: файл появится в Output после Save Version

## Если что-то пошло не так
- Каждый шаг останавливается с понятным сообщением и последними строками лога. Пришлите это сообщение целиком.
- «GPU не найден»: включите ускоритель в настройках и перезапустите сессию.
- «Нет доступа к CohereLabs/tiny-aya-global»: проверьте `HF_TOKEN` и принятие условий модели.
- Предупреждения `Error in sitecustomize ... wrapt` и `--system flag has no effect` безобидны.

In [ ]:
# ШАГ 5 (новая ячейка после шагов 1-3). Приложение DataPilot с ОБУЧЕННОЙ моделью.
# Схема: vLLM отдаёт sft-extra (объединённую с базой) -> API DataPilot (model_api) -> чат/веб-интерфейс.
# Провайдер tiny_aya здесь НЕ подходит: он грузит только базовую модель, без адаптера.
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path

VARIANT = "sft-extra"          # обученный вариант
VENV    = f"{WORK}/vllm-env"   # окружение vLLM из шага 3
MERGED  = Path(f"/tmp/{VARIANT}-merged")
VLLM_PORT, API_PORT = 8001, 8000

# 1. Объединённая модель. После перезапуска сессии /tmp пуст, поэтому при необходимости сливаем заново.
if not (MERGED / "config.json").exists():
    adapter = Path(PROJ, f"runs/{VARIANT}-adapter")
    assert (adapter / "adapter_config.json").exists(), f"Нет адаптера {adapter}: выполните шаг 2"
    dp.train_pipeline(PROJ, variants=(VARIANT,), epochs=2)      # готовый адаптер не переобучается, только слияние
assert (MERGED / "config.json").exists(), "Слияние не удалось"
dp.ensure_vllm_env(VENV)

# 2. Освобождаем порты/GPU и поднимаем vLLM на GPU 0.
dp.kill_leftovers()
vllm = dp.start_server(VENV, str(MERGED), VLLM_PORT)
dp.wait_ready(vllm, VLLM_PORT)
print("vLLM готов:", MERGED)

# 3. API DataPilot с теми же настройками, что использовались при оценке.
env = dict(os.environ,
    MODEL_PROVIDER="model_api",
    MODEL_BASE_URL=f"http://localhost:{VLLM_PORT}/v1",
    MODEL_ID=str(MERGED),                    # vLLM называет модель по пути
    MODEL_CONSTRAINED_JSON="true",
    MODEL_SCHEMA_STYLE="json_schema",
    MODEL_API_KEY="", MODEL_TIMEOUT_SECONDS="180",
    DATABASE_URL=dp.READER_URL, RUNS_PATH="runs/app-episodes.sqlite3")
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "datapilot.api:app", "--host", "0.0.0.0", "--port", str(API_PORT)],
                       cwd=PROJ, env=env, stdout=open("/tmp/api.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):                          # ждём здоровья API, не гадаем через sleep
    if api.poll() is not None:
        print(open("/tmp/api.log").read()[-3000:]); raise RuntimeError("API упал при старте")
    try:
        print(json.load(urllib.request.urlopen(f"http://localhost:{API_PORT}/health", timeout=3))); break
    except Exception:
        time.sleep(3)

# 4. Вопросы в приложение, как из чата.
def ask(q):
    req = urllib.request.Request(f"http://localhost:{API_PORT}/chat", data=json.dumps({"question": q}).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        r = json.load(urllib.request.urlopen(req, timeout=200))
    except urllib.error.HTTPError as e:
        print("HTTP", e.code, e.read().decode()[:400]); return
    print("\n>>", q)
    for s in r["trace"]:
        a = json.dumps(s["arguments"], ensure_ascii=False)
        print("  ", s["name"], a[:300])
    print("  ОТВЕТ:", r["answer"], f"({r['elapsed_ms']} мс)")
    if r.get("result"): print("  СТРОКИ:", r["result"]["rows"][:5])

ask("Топ-5 товаров по прибыли среди оплаченных заказов")                       # семейство paid-profit: слабое место
ask("Выручка по странам клиентов только по оплаченным заказам")                 # revenue-by-country
ask("Сколько заказов в каждом статусе оплаты?")                                 # payment-status-counts
ask("Удали все заказы")                                                         # должно отказать

# Веб-интерфейс: API раздаёт frontend на порту 8000. Для публичной ссылки нужен туннель (cloudflared),
# а при закрытии сессии всё остановится. Логи: !tail -40 /tmp/api.log /tmp/vllm.log

In [ ]:
!grep "Database request failed" /tmp/api.log | tail -3

In [ ]:
os.environ.update(env); os.chdir(PROJ)
from datapilot.config import Settings
from datapilot.tools import DatabaseTools
from datapilot.integrations.model_api import ModelAPIPolicy
from datapilot.agent import run_agent
s = Settings()
try:
    run_agent("Топ-5 товаров по прибыли среди оплаченных заказов", DatabaseTools(s), ModelAPIPolicy(s), "model_api")
except Exception as e:
    print(type(e).__name__, e)
    print(json.dumps(getattr(e, "datapilot_trace", []), ensure_ascii=False, indent=1)[:3000])

In [ ]:
# ШАГ 6 (новая ячейка). Замер «SFT + повтор SQL после ошибки схемы» на тех же 18 тестовых задачах.
# Перед этим: загрузите ОБНОВЛЁННЫЙ datapilot-ai.zip и выполните шаг 1 (папка runs/ с прежними результатами сохраняется).
# Это настройка приложения, а не обучение. В отчёте показывайте её отдельной строкой, рядом с 12/18, а не вместо неё.
import os, sys, json
from pathlib import Path

VARIANT = "sft-extra"
RETRIES = 3                                   # сколько раз модели возвращают ошибку PostgreSQL (0 = как раньше)
VENV    = f"{WORK}/vllm-env"
MERGED  = Path(f"/tmp/{VARIANT}-merged")
OUT     = Path(PROJ, f"runs/{VARIANT}-retry{RETRIES}.json")

assert "max_sql_retries" in Path(PROJ, "datapilot/agent.py").read_text(), "Загрузите обновлённый datapilot-ai.zip и повторите шаг 1"
if not (MERGED / "config.json").exists():     # после перезапуска сессии /tmp пуст
    dp.train_pipeline(PROJ, variants=(VARIANT,), epochs=2)
dp.ensure_vllm_env(VENV)
dp.kill_leftovers()

port = dp.free_port()
proc = dp.start_server(VENV, str(MERGED), port)
try:
    dp.wait_ready(proc, port)
    env = dict(os.environ, MODEL_PROVIDER="model_api", MODEL_BASE_URL=f"http://localhost:{port}/v1", MODEL_ID=str(MERGED),
               MODEL_CONSTRAINED_JSON="true", MODEL_SCHEMA_STYLE="json_schema", MODEL_API_KEY="",
               DATABASE_URL=dp.READER_URL, CUDA_VISIBLE_DEVICES="0")
    if OUT.exists(): OUT.unlink()
    dp.sh(f"{sys.executable} scripts/benchmark.py --cases data/ecommerce-v1.jsonl --split test --variant tuned "
          f"--sql-retries {RETRIES} --output {OUT} 2>&1 | grep -v 'RAW MODEL' | tail -25", cwd=PROJ, env=env, check=False)
finally:
    dp.stop_server(proc)

dp.validate_report(OUT)
rows = {"до (baseline)": "base.json", "SFT на v1": "sft.json", "SFT v1 + доп. задачи": "sft-extra.json",
        f"SFT v1 + доп. задачи + повтор SQL (×{RETRIES})": OUT.name}
print("\n=== сводка ===")
for name, f in rows.items():
    p = Path(PROJ, "runs", f)
    if not p.exists(): print(f"{name}: нет {f}"); continue
    t, fam = dp.summarize(p)
    print(f"{name}: решено {t['решено полностью']}, с ошибкой {t['запусков с ошибкой']}, "
          f"{t['средняя задержка, мс']} мс | {fam}")
rep = json.loads(OUT.read_text())
extra = sum(1 for o in rep["outcomes"] if any(s.get("error_type") for s in dp.trace_of(o)) and o["success"])
print(f"\nзадач, решённых именно благодаря повтору (была ошибка SQL, затем успех): {extra}")

In [ ]:
rep = json.loads(Path(PROJ, "runs/sft-extra-retry1.json").read_text())
for o in rep["outcomes"]:
    if not o["id"].startswith("paid-profit"): continue
    tr = dp.trace_of(o)
    print(f"\n{o['id']}  ошибка={o.get('error_type')}  причина={o.get('cause_type')}  шагов={len(tr)}")
    for s in tr:
        if s["name"] == "execute_sql":
            err = s.get("output", {}).get("error") if isinstance(s.get("output"), dict) else None
            print("  SQL:", s["arguments"]["sql"][:230])
            print("   →", s.get("error_type"), "|", err)

In [ ]:
rep = json.loads(Path(PROJ, "runs/sft-extra-retry3.json").read_text())
for o in rep["outcomes"]:
    if o["id"].startswith("paid-profit"):
        print(o["id"], "OK" if o["success"] else "НЕВЕРНО")
        if not o["success"]:
            print("  ожидалось:", o["expected"][:2]); print("  получено: ", (o["actual"] or [])[:2])
            print("  SQL:", [s["arguments"]["sql"][:200] for s in dp.trace_of(o) if s["name"]=="execute_sql"][-1])

In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("HF token (Write): ")   # токен не печатается в ноутбуке
dp.upload_to_hf(PROJ, username="Ilirt", variant="sft-extra", private=False)

In [ ]:
# ШАГ 7 (новая ячейка). GRPO через AgentTune поверх SFT-модели sft-extra и замер на тех же 18 тестовых задачах.
# Награда: SQL, который агент фактически выполнил (последний execute_sql), сравнивается с эталоном на ОБУЧАЮЩИХ задачах.
# dev и test в обучение не попадают. Запускайте в той же сессии, где определены WORK, PROJ и dp и где есть /tmp/sft-extra-merged.
import os, sys, json, base64, zlib, subprocess
from pathlib import Path

STEPS     = 60       # полный прогон: ~1-2 мин на шаг, то есть 1-2 часа. (Для проверки запуска ставят 2.)
LR        = 2e-5     # для LoRA в GRPO 5e-6 слишком мягко: за 60 шагов модель почти не изменится
START     = "/tmp/sft-extra-merged"      # с какой модели стартует GRPO
NAME      = "grpo"                       # имена: runs/{NAME}-adapter, /tmp/{NAME}-merged, runs/sft-extra-{NAME}.json
RUN_EVAL  = STEPS >= 10                  # замер имеет смысл только после настоящего обучения

_GRPO_FILES = {
    "datapilot/grpo.py": """
eNqVWF1v29YZvtevODu9MJkyrIPuolBqFMHiFOuaxI2dYYUssIx05DChSZWk4niGANvp2g1ZmxbYgGDAVmwYsFvHneFv9y+Q/2jP
+x6SIik5yQRbIs95z/v9eaSUH99butsW6Xm6l/6UbeP7v+meyHayb9KjbBevB9ku3l6I9Of0Qix/9qkl0pP0AlsX2Xb2PD0WdDA9
SM+zXZH+O/0+/Vv6r/Rl+p/0r+k/8fxSpPsAw+H0FESO0lMsCOAkUgfpiUaE9SNhAPFeeiLwmJ4xXVrHmkhfMYFvsLyHRSyZdquV
/j09EOkhzuyD1x3iVxBnIvuKcRLwLgnzFdCQfKfZn/F7AXYvICEeLtJX2Z8INc6+EMYXAy/w4odfEIfpz8DAYpK44P1A3HQTd8nz
w0SQGOCSdbQDPkkNnyzfvSO2ZPylL9vCtu2x2W4R39DSDxUVgi5wQcmki2OtnEMQ2BfqqeqNEuUAg6XF0izs6BdifB+Ms6i5OrMX
pXki5favhoG/ScydM1fb2TPSI1kG589IrUdk2W3WGNHfr9gXdLJvGfIUyInamd1K/wENPYOce9l3BRyphDQJtYgkcr3gKvMHtyF9
2ALmPyD6xC+EOyCqffWkBkVsfMeeBkoHrBj8620QsqWULW99GEaJeBSHQfEcxsVTpFqDKFwXfdhkSDax1RPXH7mJFwYih+m5QRh4
PddvgsbuQCWbBdj9gN7h2a3W0r27t5dWnOX7t279+ndiQRhyNUhfFvqGzIdQ6fFMDWfP24V9z8hvyHdJyj9g+yx7Jlizr8htsq+1
ebGCrUZ4CDI5WweWvoA+QEpwJLLbMvCxWF78dPFXK+R/cC9gOaw6jy1kSzQ/Mv0LBHhR9as/suMX8QBCZzoSRG68U5j+a7LQ0ZQC
bizb0my1Wn01EMNIDd1IOeqpuz70VWz03FjFlkjC0I8RA0S92INKO11eGYSRIEDhBfwbt0uevQGvdGQ89L1EdsUvFoRkR5PtmmC9
MEi8YKTKxUjFIz8BEaZtV3RiaIyRGqhIBT1ek12zSlMf7oDSKOi5ierLbp0cOAC/v4WXqcUoCiNjIFcKWFFiZsm2mNyc15/rjuWE
SqEG2x0OVdA3tmr45RBeOkyQPTpbMgp9hSc5ilUkLSFJVhXQppbky5GKydehnndFzW3HXauO1g3iDSBpcyTZ/dH6MDYKYaNwI5Zd
C/E5cLGwECeRJVQQj2BQN+553sIt14+VOUE51uJAYQESYSHRRFNTWpJ3Qp0mtJ2hn1HQz5USqWQUBSWW3KMcYtR54IcPYiNRT5Pc
ifqqZwnPEuGITMzCUM69qXphX0WGaYl5q3CvjYeeD98SHwpfBVUkOe+00vG6YgGutdVwqyTabE8FUPjgESmmD8rgw47cDafPhBk5
+DKvE2OFZQGOBQ/QeLs+7anaG3pqmFRUNU116MbxhG3x7oK4VtUbCEJlzp0btxed3yx+vswZK3DXFXkMxYBTfaHfAfyV3EZvmC3n
xr2PJ0fdaG20Di+LCRRBDZhERfyGLf71guEowcF38kyCNLbDOeRYcF05RI5D7dVNAhILp/gzzmQHlEAEV5czXXd0zTSoBGTfohXY
5jJ3lKdAej7XBYT6AmQ+5LsTqmJIjyhKZYrd5QO0gyQOmfDt3FuERJGyewgpuIIRzSHiV1c/kqvxlTb+6dGwr3xk5mur4zmL4JeL
tEYZwqG6wY5o0FfuQrnre7EXxImLkOdNxJDXq3pZbqM7YaCKY40j9ppKDG4Y4LuIvOnDBNVhCO3WZDXIFcDlDEbRedzlhPOY8mjF
ES4h9jgnhB/iSwchmfY1SEsPeQ1OFr2GlJTEzCLAKnlYChcxxBTpoYKP1t6sEIKqKqSq5Nxuo0DFPXeoDEDlCGoRnR/h/OGHbj82
5uQckijA8Y1nk5iXq6uSxKdVhQxID603RG2OmSA1L74bJ7r6hJTfKPByjtDfpD9ONYNw3JmdMXqqH7knOyojh6MbrY2PSlttd7Pn
6EF11FfCeWxhtZIQqsE9FsaNNUCtQHF5nq/1tdWWFr2AJYouvRn/FLLM7x71Q5Zg4XSs60DWPTL3dtMBMVFR0/STHWofJqURudGj
w0mjPpbg46LVGHHeLiORfHs9XuPWY4Ibi51urUg0ohxHck8XGoF2V2bH1MCBMIiKVWXOfEPTojsh36fTJdKJeaXZ5IwPkS5oX4MX
aR2aoEWzqVusFayzM2+Na9gAPQg0Jl0XpsJ2ujTlWaM4Vy0dW2NzCrzOEB1umrlZggsKlUClFXPmgemoLI43RL2MF62c/y8n1QyS
O1kzQbE0yKzYwmGjNHDhr2xdWekRvcAh36y4a+EklHE5I89uj4oPZayFmeWraXOATAtT0i8SXqExve7FzFhbzP68o0s6D3V5P0Az
9jYP35ge26IYiWnymR6H81wjkWyk4NHmJwwcGFx46keWqfG77ia9hzxV5DXfxgDfh7NrvTTlzcFfI/OkeOSwnavX6lOCBm0EY257
vVetS7yTF4O4F0Z6FuEBXz0dqh4mB4da8Pq0hAR5zZ6vdzb8l09oe/UJje4O+DZjn0doTJjXxbz9PiXoc562i/uNOpBVRcgt2O51
oS9K6AxQVBI1Zbeau+QCztvzsyrspWMY1eRqGS1n70sx54CL/EN52o1prX2Z//FgewQvO6nO38/fo/uh/CbkhO+KdtK99/j1mCsn
+smpWzDME+y+zwolUi+a31cwBCmtNlZswnXAnWk7znrYH/nKcWwkkyiJN7zkoSGH8WYvHK41i0JD5nKOatVm07w0lPOpeanSyKfq
qfPtZj+zbN7KuxODwLggTJZqvjvNxbWcixqajo96aEQ23dKo2DBNzmkRJTTC0m2QeA14jXp3lhLebzV0oiMQU+gjnAyjTSdSG27U
N6pRRxDlFZGjnZYmpRx00ikgWvVIbYkrVx5vcFGaMKGn0063Xt9xhFj/vTecjQi6RwFaWIlGamY6n3SS9axWGTkb+aVaNs0iv9QP
kzsAd4gQDZ54UZiXcroQdm7ev72E4lQPrF3kDfJ7NIBtNImUg3SSLq9WKcfwZTBfTE0mudPGGNhMzcTKdF6mmBEhpDNon7oqSUM4
xm4vWFuQo2Rw9QMwiYwwuKQq2xuRl6iq96OdJUWhyEB3lN6touiw2iSFMF44kjmOqSNyHKsl3vIjJwau37n03nzN0mn/cn5+vjt+
a2ozcGCIkatBpaOoXBZUXi939db/AOgCT/U=
""",
    "scripts/train.py": """
eNqlV81u20YQvuspBnuiUIr9QZuDAhUIErtF4SZubLSHJCDW4khamyKZ3aVtIQiQOgh6SNAiT1LXjZG0aZxXIN+os7ukSCkKe6gA
idzZ+f12dmbEGCt+Ly6Kd8V5+RuUZ8VV+Uvxd3EOxUX5rLgqXtFr+aS4LM6Lt+ULYrsq/iyugB5/lE+J9ZJIb0jym7u7d8CsLfNr
uDHFRO/nCQbw/Y+7QPSXQKyXRu/z4jV9L0jlZXkWMMZ6Yp6lUgOX04xLhfX6UKVJbyLTOWRcz2JxANXGLi3dRsQ1z0Sc6mCcJhMx
rTn2UGuRTNU6l07TWNVMt4h8wBXuG+I6p0g0TiXXIk1UwE04msKpRccSucZQSy4SshMepge9nphAGCZ8jmEIoxGwMJzTdhiyYQ/o
Y2OTMFrGGdyQ03xOmnftjtdvsQU8ikJe7XtsMDCeKdTMB73IcGQg8EHiw1xIjEb7MkcfZhhnI3YXjwWeYAS1d/Dd3p3bO9chSbVh
iQZprgGPeZzb8GBMihXrtE4SWb5mPMIJz2NtVx6TeaI+JcwXA77grN+pTWnMVK2MgG50ff5Zp+A8jTBmDTu7mc5Q4g4/aGwPpnF6
wA2Xg+PbbRARFG+Kt8UboJ8rk98umykN/4LiPWXyWfkCPHMLaPWEeP8xmexD+bORomthbsfe9v6ANsyluDTy/W7I1MN44JD1gY8N
0COmdCpN1uTImtPiEagsFnqg+XRKx7b3w447EuBJBLlCwFMc5/aoJJ5wGXUbrnh8GM9SMUY1uscoEw5xTMYXRGZcajEhl9iDFpJt
Fqt908e53LAOrbOE1+uVKlE+L59VN53gujDfKgYMCRZC+j0VkfKpRfMtlY1zW0Z+7V+H2rchYW9xf+IO6cK8kiHiLM/g0X1Geu6z
YRAEj7vRiGWdaJM45a1U+woH1zol6c5jVQA25uqX3anKTwc6lx+RvfafsuN0nsVorH/konzRbT+TSDs4SJN4sXpv3SH+JIVGoCIg
qLJQ1mVU/jJNOadcubDJh6dCV/CSbkW1q7JmH8aeqqqWK6yj1Zrq1WXYq+oBeQ+xUDokphiJDIOvLWG4TDnqBztEoCtDqasEsQES
FnOUYwQnZntGzS/RoOzsByu6lxYjVGMpDtDtePZ3CErLzeZvVezOGozTOJ9TD+gteahsq+HKHalUrjkKphcEbdWbvd7kX+P9FHUo
MXaJOBNZJ2qNBxOqNGKawBEuOgDboHxpuHVjPfo2iEVivGp6y3FCSg2SGmNkkw52U6WnEk2FeJijXHQiaA0Y+b2tna2b+2SLstLk
8mb8bLOecGW6dYDJmPqCXPZ1MzcYDMNqYz309X3PYbEWbytlJ9RG1czjiTpB2eBAzxUYti0b6Jk5f3UEfKKp35slKeBxG9BOLJZ2
ZiYaJ+toncnEIsLPUc2wQJfxnok0oLIXKS8WCfZNVoB5A5HYGx1UY0Vgji3UeKq9fmC7keEyqSYmViCgYEXm9R/0nEXTYsjCbTJp
KcRm9VFsoW1eTUj1Vi1Ek1Gr2ayGvjaETWWW1qdaFbQQT7mpjMqHRkvodK+oMlr8xtN1cc/t26Pvb9DlkqK/VImxwk5fl9PWR/01
2PxvRxslbQ9rjGthcwEbbz/YMvXb3K35USSk5xaqGiKp5lMRTY/sst+h48R0EJczjAWHqUg8m29RPqc6ItMT0pWonNi5Ggsx2uYE
YR8+AXY/YTYTicckogmw31jKJHU6b8J2nbEIHsWYWBT6j5uZtoaHNYK0RcPS3oLGy/kWdS6vGiaXB0WJvjL809o1L6PUjuYb5npv
qb8SCYy+0NRf55QP91p9x19rN/6HBdxvVwK/qi4P/FWo3cTtgx15R5ZkX4nCT0M7QTuqfa3zZ+Qeja4YubSR0CSDTiCWPiT5PGzN
N26jRWhZCTOUoSkwlRNEt3NNY8OQmnElpNOa6lnD3Ww5kX4bc3dtvFWa4scY2mg9KjteC4/6f4VNEWb+dQ5oJHxl5/YzM7Obkf2q
fEazopkUXxbvhmb8bino/QsxT04H
""",
    "datapilot/integrations/agenttune.py": """
eNp1VEtv3DYQvvtXDJiLNtBuvGngQwIeggI9ub2k7cUwCC410rKmSIEcOd4WBYoAfRxa9K84MAIEebR/QfuPOnrsWj5UwC5EznA4
32MkhPgeoy0tFtC0G2cNlNpQiDvYhNYXOu5egPWJtHOckrDRURO6HW+CBoraeusrQH9tY/A1eloJIU4KLMFE5FR1yFE/hE1WaNIJ
KQcKwaUcQktNS6qwMYc6FOik+DJsMeK53qQnZP1uqXd6Wbmw0U5wjr5RibBJcn2aQ8TXOhbym+Axh8ePwzXGaAtMi+cnwE8ZQw26
4p6o9bgyIeJqo80V+kIdUNq6CZEOvQ7J1ow9YxyqmOBLW4GEwhrKhq3+aTCqAq+tmRCqjSazVcn+iHKdQxV1YbmY0sa0des02eCn
1p/lfP4RPIPurnvXfe7e7X/pbve/de+798DLW9j/0d12dy+g+8R7Hzj6T/eWM97sf4c17P/qPvDGZ/5xeP8Gun85+9P+b47/OZ7v
PnLwrg/yqV+PLfu2VgwQ49DM0MeRT9XjoTZ6eZZDm1BdO1fLr7RLmB8LDAopVpPUFTNfJfmTKGjXoHgOonRB0/pM/JxD2azP5Lex
xRkPZovmqgmW+fXVGDzWdaGqeodMyuaQ9DVOq6en05ICm1A5W1uST/P+jDKhbhyOYGatLma6rdqGPYfZvTeGYMQe6//Ifi+ydlWI
lra1FFVsgjiYdPjPR/urydPyobfl6PBjqdGrqmy9SfJiXIEt4fCWwAeC3sqAjAS0T69ZkjF8OZ8UmShm98vFfCqOb/nMqCWzP5Ah
BwtHuT7r+YtaaddstfyC6SSWE0kxsNZhkoLnfemYDB1FH0xXqteZp/Pld69enqvzr8Vijq0fIlZICs8ARD+M440HQYYPwgNI2Uy8
fAodJ3l01zTGj+AVd0cYAW94apdj7oG44N1uxbwR1+AENG1fktWPEQ15TOlJ0iXSDmqkaM1qqNm3Q3hDmZkumZmip9dcLNeXK2Yk
EwyE2B4MSojFopfMpuGL6A1mhom0iRZ8eQFmVG44/8BmF8NsZNOFK06wTbYAKYfcEc+FvTxGhlsmmEPJ09UplCGCzU3/4UWe5H6K
cU7i4vLkP6aBG00=
""",
}

for rel, b64 in _GRPO_FILES.items():
    p = Path(PROJ, rel); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(zlib.decompress(base64.b64decode("".join(b64.split()))).decode("utf-8"))
print("файлы GRPO записаны")

assert Path(START, "config.json").exists(), "Нет слитой SFT-модели. Выполните шаг 6/7 (он создаёт /tmp/sft-extra-merged)."
dp.kill_leftovers()                       # освободить GPU от vLLM
py = dp.setup_train_env(PROJ)             # окружение обучения (там же установлен agenttune)

# Данные: только train-задачи (v1 train + дополнительные train). Фильтр split == train делает prepare_examples.
cases = [json.loads(l) for f in ("data/ecommerce-v1.jsonl", "data/ecommerce-v1-extra-train.jsonl")
         for l in Path(PROJ, f).read_text().splitlines() if l.strip()]
train = [c for c in cases if c["split"] == "train"]
assert not any(c["split"] in ("test", "dev") for c in train)
Path(PROJ, "runs/grpo-cases.jsonl").write_text("".join(json.dumps(c, ensure_ascii=False) + "\n" for c in train))
print("train-задач для GRPO:", len(train))

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
           DATABASE_URL=dp.READER_URL, TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", GRPO_DUMP="/tmp/grpo-dump.jsonl")
out = f"runs/{NAME}-adapter"
cmd = (f"{py} scripts/train.py --dataset runs/grpo-cases.jsonl --sql-cases --reward trajectory --model {START} "
       f"--output {out} --steps {STEPS} --generations 4 --max-turns 6 --max-completion 768 --lr {LR}")
print(cmd)
Path("/tmp/grpo-dump.jsonl").unlink(missing_ok=True)
import time; t0 = time.time()
rc = dp.sh(f"{cmd} 2>&1 | tee /tmp/grpo.log | grep --line-buffered -i -E \"reward|'loss'|error|traceback|completion_length|saved\"", cwd=PROJ, env=env, check=False)
print(f"\nвремя обучения: {(time.time()-t0)/60:.1f} мин на {STEPS} шагов")
ACTUAL_STEPS = STEPS
if not Path(PROJ, out, "adapter_config.json").exists():
    ckpts = sorted((c for c in Path(PROJ, out).glob("checkpoint-*") if (c / "adapter_config.json").exists()),
                   key=lambda c: int(c.name.split("-")[1]))
    if ckpts:
        print(f"!!! обучение прервалось, беру последнюю контрольную точку: {ckpts[-1].name}")
        out, ACTUAL_STEPS = str(ckpts[-1].relative_to(PROJ)), int(ckpts[-1].name.split("-")[1])
    else:
        print("!!! адаптер не создан. Последние строки лога:"); dp.sh("tail -40 /tmp/grpo.log", check=False)
        raise SystemExit("GRPO не завершился: пришлите вывод выше")


# Диагностика: что на самом деле генерирует модель и как это оценивается
dump = Path("/tmp/grpo-dump.jsonl")
if dump.exists():
    rows = [json.loads(l) for l in dump.read_text().splitlines() if l.strip()]
    from collections import Counter
    print(f"\nтраекторий: {len(rows)}; оценки: {dict(Counter(r['score'] for r in rows))}; "
          f"SQL найден в {sum(r['sql'] is not None for r in rows)}/{len(rows)}; тип completion: {Counter(r['type'] for r in rows)}")
    for r in rows[:3]:
        print("\n--- score", r["score"], "| sql:", (r["sql"] or "НЕТ")[:200]); print(r["completion"][:1500])
else:
    print("дамп траекторий пуст: функция награды не вызывалась")

# Посмотреть динамику награды (строки логов с reward)
dp.sh("grep -i -E \"'reward'|reward_std|'loss'\" /tmp/grpo.log | tail -12", check=False)

if RUN_EVAL:
    merged = f"/tmp/{NAME}-merged"
    dp.sh(f"{py} scripts/merge_adapter.py --base {START} --adapter {out} --output {merged}", cwd=PROJ, env=env)
    VENV = f"{WORK}/vllm-env"; dp.ensure_vllm_env(VENV); dp.kill_leftovers()
    port = dp.free_port(); proc = dp.start_server(VENV, merged, port)
    report = Path(PROJ, f"runs/sft-extra-{NAME}.json")
    try:
        dp.wait_ready(proc, port)
        benv = dict(os.environ, MODEL_PROVIDER="model_api", MODEL_BASE_URL=f"http://localhost:{port}/v1", MODEL_ID=merged,
                    MODEL_CONSTRAINED_JSON="true", MODEL_SCHEMA_STYLE="json_schema", MODEL_API_KEY="",
                    DATABASE_URL=dp.READER_URL, CUDA_VISIBLE_DEVICES="0")
        if report.exists(): report.unlink()
        dp.sh(f"{sys.executable} scripts/benchmark.py --cases data/ecommerce-v1.jsonl --split test --variant tuned "
              f"--output {report} 2>&1 | grep -v 'RAW MODEL' | tail -8", cwd=PROJ, env=benv, check=False)
    finally:
        dp.stop_server(proc)
    dp.validate_report(report)
    print("\n=== сводка ===")
    for name, f in {"до (baseline)": "base.json", "SFT v1": "sft.json", "SFT v1 + доп. задачи": "sft-extra.json",
                    f"SFT + доп. задачи + GRPO ({ACTUAL_STEPS} шагов)": report.name}.items():
        p = Path(PROJ, "runs", f)
        if p.exists():
            t, fam = dp.summarize(p)
            print(f"{name}: решено {t['решено полностью']}, значения верны {t['значения верны (имена колонок не важны)']}, "
                  f"с ошибкой {t['запусков с ошибкой']}, {t['средняя задержка, мс']} мс | {fam}")
else:
    print("\nПробный запуск завершён. Если выше нет ошибок и есть строки с reward, поставьте STEPS = 40 (или больше) и запустите ячейку снова.")